### Sarvam AI STT

In [ ]:
#Download audio via URL

import subprocess

url = "https://www.youtube.com/watch?v=68ylaeBbdsg"
title = "sarvam_test_audio"
subprocess.run([
    "yt-dlp",
    "-x", "--audio-format", "mp3",
    "--audio-quality", "0",
    "-o", f"./{title}.%(ext)s",
    url
])

In [ ]:
#Hear audio to verify proper download

from IPython.display import Audio
Audio("sarvam_test_audio.mp3")

In [ ]:
#Transcribe audio using Sarvam API

from sarvamai import SarvamAI

def main():
    client = SarvamAI(api_subscription_key="sk_xf3bnhg0_3uTPsVDnhH632eFAp6eGHExM")

    # Create batch job — change mode as needed
    job = client.speech_to_text_job.create_job(
        model="saaras:v3",
        mode="transcribe",
        language_code="unknown",
        with_diarization=True,
        num_speakers=2
    )

    # Upload and process files
    audio_paths = ["./sarvam_test_audio.mp3"]
    job.upload_files(file_paths=audio_paths)
    job.start()

    # Wait for completion
    job.wait_until_complete()

    # Check file-level results
    file_results = job.get_file_results()

    print(f"\nSuccessful: {len(file_results['successful'])}")
    for f in file_results['successful']:
        print(f"  ✓ {f['file_name']}")

    print(f"\nFailed: {len(file_results['failed'])}")
    for f in file_results['failed']:
        print(f"  ✗ {f['file_name']}: {f['error_message']}")

    # Download outputs for successful files
    if file_results['successful']:
        job.download_outputs(output_dir="./output_sarvam")
        print(f"\nDownloaded {len(file_results['successful'])} file(s) to: ./output_sarvam")

if __name__ == "__main__":
    main()

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()


In [ ]:
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env")

In [ ]:
#Test gemini api

from google import genai
from google.genai import types

gemini_client = genai.Client()

response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        system_instruction="You are a class 5 school student. Your name is Neko.",
        temperature=0.1
        ),
    contents="How does AI work? Respond in 50 words"
    )

print(response.text)

### Add Speaker Names to Conversation

In [ ]:
import json
import re
import os


SPEAKER_MAP = {
    "0": "Nikhil Kamath",
    "1": "Dario Amodei"
}

# load raw sarvam transcript 
with open("./output_sarvam/sarvam_test_audio.mp3.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

entries = raw.get("diarized_transcript", {}).get("entries", [])
print(f"Total raw entries: {len(entries)}")

In [ ]:
# apply speaker mapping
mapped = []
for e in entries:
    sid = str(e.get("speaker_id", "0"))
    name = SPEAKER_MAP.get(sid, f"Speaker_{sid}")
    mapped.append({
        "speaker": name,
        "text":    e.get("transcript", ""),
        "start":   e.get("start_time_seconds", 0),
        "end":     e.get("end_time_seconds", 0),
    })

# preview first 5 entries
for e in mapped[:5]:
    print(f"[{e['speaker']} @ {int(e['start'])}s] {e['text'][:80]}...")

In [ ]:
# merge mid-sentence continuations into previous entry
merged = []
for e in mapped:
    first_word = e['text'].strip().split()[0] if e['text'].strip() else ""
    is_continuation = (
        e['text'] and e['text'][0].islower()  
        or len(e['text'].split()) < 4          # very short fragment
    )
    if is_continuation and merged and merged[-1]['speaker'] == e['speaker']:
        # glue onto previous entry from same speaker
        merged[-1]['text'] += " " + e['text']
        merged[-1]['end'] = e['end']
    else:
        merged.append(dict(e))

print(f"Entries before merge: {len(mapped)}")
print(f"Entries after merge:  {len(merged)}")

words_merged = sum(len(e['text'].split()) for e in merged)
print(f"Words preserved:      {words_merged:,} / {13008:,} ({words_merged/13008*100:.1f}%)")

In [ ]:
# save speaker mapped transcript 
os.makedirs("checkpoints", exist_ok=True)
with open("./checkpoints/speaker_mapped_merged_transcript.json", "w", encoding="utf-8") as f:
    json.dump(mapped, f, ensure_ascii=False, indent=2)

print(f"Saved {len(mapped)} speaker mapped merged entries")
print(f"Speakers found: {set(e['speaker'] for e in mapped)}")

In [ ]:
# config 
BATCH_SIZE = 80

# build batches
batches = [merged[i:i+BATCH_SIZE] for i in range(0, len(merged), BATCH_SIZE)]
print(f"Total batches: {len(batches)}  ({BATCH_SIZE} entries each, last batch may be smaller)")

In [ ]:
# clean each batch 

import time

def clean_batch(model, batch, batch_num):
    raw_text = "\n".join(
        f"[{e['speaker']} @ {int(e['start'])}s] {e['text']}" for e in batch
    )

    prompt = f"""You are preparing a raw interview transcript for a non-fiction book.
The two speakers are Nikhil Kamath (host) and Dario Amodei (CEO of Anthropic).

TASK: Lightly clean the transcript. Be CONSERVATIVE — when in doubt, keep the text.

REMOVE only:
- Pure filler sounds: um, uh, hmm (only when standalone, not mid-sentence)
- Exact stutters: "I I I" → "I", "So so" → "So"
- Obvious off-record lines like "was that okay?", "are we recording?"

DO NOT REMOVE:
- Any sentence with a real idea, opinion, or fact
- Short answers like "yes", "no", "exactly", "right" — these show agreement
- Incomplete sentences — keep them as-is
- Anything you are unsure about — KEEP IT

CRITICAL RULES:
- Do NOT summarize or compress any speech
- Do NOT merge entries unless they are identical repetition
- Every speaker turn must stay as a separate line
- Return in EXACT same format: [SpeakerName @ Xs] text

Return ONLY the cleaned transcript. No commentary.

TRANSCRIPT:
{raw_text}"""

    print(f"  Calling Gemini for batch {batch_num}...")
    response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=prompt
    )
    time.sleep(4) #free tier rate limit
    return response.text

In [ ]:
# re-run cleaning with new prompt
all_cleaned_text = ""
for i, batch in enumerate(batches):
    result = clean_batch(gemini_client, batch, i+1)
    all_cleaned_text += result + "\n"
    print(f" Batch {i+1}/{len(batches)} done")

print(f"\n All batches cleaned")

In [ ]:
# re-parse
cleaned_entries = []
for line in all_cleaned_text.strip().split("\n"):
    line = line.strip()
    if not line:
        continue
    m = re.match(r'\[(.+?)\s*@\s*([\d.]+)s\]\s*(.*)', line)
    if m:
        cleaned_entries.append({
            "speaker": m.group(1).strip(),
            "start":   float(m.group(2)),
            "text":    m.group(3).strip(),
        })
    else:
        if cleaned_entries:
            cleaned_entries[-1]["text"] += " " + line

print(f"Entries before cleaning : {len(merged)}")
print(f"Entries after cleaning  : {len(cleaned_entries)}")

# check word count again
words_before = sum(len(e['text'].split()) for e in merged)
words_after  = sum(len(e['text'].split()) for e in cleaned_entries)

print(f"Words before: {words_before:,}")
print(f"Words after:  {words_after:,}")
print(f"Difference:   {words_before - words_after:,} words ({(words_before-words_after)/words_before*100:.1f}%)")
    

In [ ]:
# find which entries exist in merged but not in cleaned
merged_texts = set(e['text'][:50] for e in merged)
cleaned_texts = set(e['text'][:50] for e in cleaned_entries)

dropped = [t for t in merged_texts if t not in cleaned_texts]
print(f"Possibly dropped {len(dropped)} entries:\n")
for d in dropped[:10]:
    print(f"  → {d}")

In [ ]:
# also check average entry length before vs after
avg_before = sum(len(e['text'].split()) for e in merged) / len(merged)
avg_after  = sum(len(e['text'].split()) for e in cleaned_entries) / len(cleaned_entries)

print(f"Avg words per entry before: {avg_before:.1f}")
print(f"Avg words per entry after:  {avg_after:.1f}")

In [ ]:
# check how many original merged entries start without proper content
short_entries = [e for e in merged if len(e['text'].split()) < 5]
print(f"Very short entries (under 5 words): {len(short_entries)}")
for e in short_entries[:10]:
    print(f"  [{e['speaker']} @ {int(e['start'])}s] {e['text']}")

In [ ]:
# check entries that start mid-sentence (lowercase first word)
mid_sentence = [e for e in merged if e['text'] and e['text'][0].islower()]
print(f"\nEntries starting mid-sentence: {len(mid_sentence)}")
for e in mid_sentence[:10]:
    print(f"  [{e['speaker']} @ {int(e['start'])}s] {e['text'][:80]}")

In [ ]:
# preview a few cleaned entries 
for e in cleaned_entries[:5]:
    print(f"[{e['speaker']} @ {int(e['start'])}s] {e['text'][:100]}...")

In [ ]:
# save checkpoint 
checkpoint = {
    "entries": cleaned_entries,
    "raw_cleaned_text": all_cleaned_text
}

with open("checkpoints/1_cleaned_transcript.json", "w", encoding="utf-8") as f:
    json.dump(checkpoint, f, ensure_ascii=False, indent=2)

print(f"Saved: checkpoints/1_cleaned_transcript.json")
print(f"Total cleaned entries: {len(cleaned_entries)}")

### Load Cleaned Transcript to LLM to extract chapters and relevant content

In [ ]:
with open("checkpoints/1_cleaned_transcript.json", "r", encoding="utf-8") as f:
    data = json.load(f)

cleaned_text = data["raw_cleaned_text"]

print(f"Loaded cleaned text length: {len(cleaned_text)}")

In [ ]:
chapters_prompt = f"""
You are an expert non-fiction book editor.

Your job is to convert a cleaned interview transcript into structured book chapters.

Instructions:
- Group content by TOPIC, not by timeline.
- Combine related ideas even if they appear in different parts of the transcript.
- Create 8 to 10 chapters.
- Each chapter must have:
    1. A strong title
    2. A short theme description (1 line)
    3. Full content (merged and organized text)

Rules:
- Do NOT summarize too much — retain original meaning
- Do NOT invent new ideas
- Keep author voice intact
- Make content readable and organized

Return ONLY valid JSON in this format:
{{
  "chapters": [
    {{
      "title": "...",
      "theme": "...",
      "content": "..."
    }}
  ]
}}

Transcript:
{cleaned_text[:200000]}
"""
print(f"  Calling Gemini for extracting chapters...")
response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=chapters_prompt
    )
time.sleep(4) 
raw_output = response.text
print(raw_output[:1000]) #preview


In [ ]:
#Clean raw json output of Gemini LLM

import re
import json

def extract_json(text):
    # Remove markdown fences if present
    text = re.sub(r"```json", "", text)
    text = re.sub(r"```", "", text)

    # Extract JSON block safely
    match = re.search(r"\{.*\}", text, re.DOTALL)
    
    if match:
        json_str = match.group(0)
        return json.loads(json_str)
    else:
        raise ValueError("No valid JSON found in Gemini output")



In [ ]:
chapters_data = extract_json(raw_output)

In [ ]:
print(len(chapters_data["chapters"]))

In [ ]:
for ch in chapters_data["chapters"]:
    print(ch["title"])

In [ ]:
for i, ch in enumerate(chapters_data["chapters"], 1):
    print(f"\nChapter {i}: {ch['title']}")
    print(f"Theme: {ch['theme']}")
    print(f"Content length: {len(ch['content'])}")

In [ ]:
#Save checkpoint of chapters wise content output

with open("checkpoints/2_chapters_wise_content.json", "w", encoding="utf-8") as f:
    json.dump(chapters_data, f, ensure_ascii=False, indent=2)

print("Saved: checkpoints/2_chapters_wise_content.json")
print(f"Chapters created: {len(chapters_data['chapters'])}")

### Chunking text of each chapter to fit context window

In [ ]:
import nltk
nltk.download('punkt') #Pre-trained sentence splitter
from nltk.tokenize import sent_tokenize

In [ ]:
#Chunking Function

def chunk_text(text, max_words=3000, overlap_sentences=2):
    sentences = sent_tokenize(text)
    
    chunks = []
    current_chunk = []
    current_word_count = 0

    for sentence in sentences:
        word_count = len(sentence.split())

        if current_word_count + word_count > max_words:
            chunks.append(" ".join(current_chunk))

            # overlap: take last N sentences
            overlap = current_chunk[-overlap_sentences:] if overlap_sentences > 0 else []
            current_chunk = overlap.copy()
            current_word_count = sum(len(s.split()) for s in current_chunk)

        current_chunk.append(sentence)
        current_word_count += word_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [ ]:
#Apply chunking function to chapters

chunked_chapters = []

for ch in chapters_data["chapters"]:
    chunks = chunk_text(ch["content"])

    chunked_chapters.append({
        "title": ch["title"],
        "theme": ch["theme"],
        "chunks": chunks
    })

print(f"Total chapters: {len(chunked_chapters)}")
print(f"Chunks in first chapter: {len(chunked_chapters[0]['chunks'])}")

In [ ]:
#Save checkpoint of chunked chapters

with open("checkpoints/3_chunked_chapters.json", "w", encoding="utf-8") as f:
    json.dump(chunked_chapters, f, ensure_ascii=False, indent=2)

print("Saved: checkpoints/3_chunked_chapters.json")

### Rewrite dialogues into book prose

In [ ]:
# Load data

import json

with open("checkpoints/3_chunked_chapters.json", "r", encoding="utf-8") as f:
    chunked_data = json.load(f)

print(f"Loaded chapters: {len(chunked_data)}")

In [ ]:
#Prompt

def build_rewrite_prompt(title, theme, chunk_text):
    return f"""
You are a professional non-fiction writer.

Rewrite the following interview content 'Text' into smooth, engaging book prose.

Context:
Chapter Title: {title}
Theme: {theme}

Instructions:
- Convert dialogue into narrative paragraphs
- Remove filler words (um, uh, etc.)
- Keep original meaning EXACT
- Maintain natural storytelling flow
- Do NOT summarize too aggressively
- Expand explanations where needed. Do not compress ideas.
- Remove all conversational markers like "he said", "she said", etc.
- Quotes must be impactful and directly usable in a book.

Also extract:
1. 2-3 strong quotes (verbatim, impactful lines)
2. Any statements that may require fact-checking like fact_flags

Return ONLY valid JSON:

{{
  "prose": "...",
  "quotes": ["...", "..."],
  "fact_flags": ["...", "..."]
}}

Text:
{chunk_text}
"""

In [ ]:
#Process each chunk

import time

rewritten_output = []

for ch in chunked_data:
    print(f"\n Processing Chapter: {ch['title']}")

    rewritten_chunks = []

    for i, chunk in enumerate(ch["chunks"]):
        print(f"   Chunk {i+1}/{len(ch['chunks'])}")

        rewrite_prompt = build_rewrite_prompt(ch["title"], ch["theme"], chunk)

        print(f" Calling Gemini for extracting chapters...")
        response = gemini_client.models.generate_content(
            model="gemini-3.1-flash-lite-preview",
            config=types.GenerateContentConfig(
                max_output_tokens=4096),
            contents=rewrite_prompt )

        try:
            parsed = extract_json(response.text)
        except Exception as e:
            print("JSON parse failed, saving raw")
            parsed = {
                "prose": response.text,
                "quotes": [],
                "fact_flags": []
            }

        rewritten_chunks.append(parsed)

        time.sleep(2)  # avoid rate limits

    rewritten_output.append({
        "title": ch["title"],
        "theme": ch["theme"],
        "rewritten_chunks": rewritten_chunks
    })

In [ ]:
#Save checkpoint

with open("checkpoints/4_rewritten_chapters.json", "w", encoding="utf-8") as f:
    json.dump(rewritten_output, f, ensure_ascii=False, indent=2)

print("Saved: checkpoints/4_rewritten_chapters.json")

### Build full chapter drafts

In [ ]:
#Load Data

import json

with open("checkpoints/4_rewritten_chapters.json", "r", encoding="utf-8") as f:
    rewritten_data = json.load(f)

print(f"Loaded chapters: {len(rewritten_data)}")

In [ ]:
#Build full chapters prompt 

def build_chapter_prompt(title, theme, combined_text, quotes):
    quotes_text = "\n".join([f"- {q}" for q in quotes[:3]])

    return f"""
You are a professional non-fiction book editor.

Create a polished, publish-ready chapter.

Chapter Title: {title}
Theme: {theme}

Content:
{combined_text}

Quotes to consider:
{quotes_text}

Instructions:
- Write a strong engaging opening hook
- Merge content into smooth narrative flow
- Expand explanations and preserve richness of ideas.
- Insert 1-2 quotes as block quotes (use markdown > format)
- Maintain logical progression
- Ensure smooth transitions between sections.
- End with a reflective closing paragraph

Return ONLY the final chapter text.
"""

In [ ]:
#Process each chapter

import time

final_chapters = []

for ch in rewritten_data:
    print(f"\n Building Chapter: {ch['title']}")

    # Combine all prose
    combined_text = "\n\n".join(
        chunk["prose"] for chunk in ch["rewritten_chunks"]
    )

    # Collect quotes + fact flags
    all_quotes = []
    fact_flags = []

    for chunk in ch["rewritten_chunks"]:
        all_quotes.extend(chunk.get("quotes", []))
        fact_flags.extend(chunk.get("fact_flags", []))

    prompt = build_chapter_prompt(
        ch["title"],
        ch["theme"],
        combined_text,
        all_quotes
    )

    response = gemini_client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=4096,
        )
    )

    chapter_text = response.text

    final_chapters.append({
        "title": ch["title"],
        "theme": ch["theme"],
        "chapter_text": chapter_text,
        "all_quotes": all_quotes,
        "fact_flags": fact_flags
    })

    time.sleep(2)

In [ ]:
#Save checkpoint

with open("checkpoints/5_final_chapters.json", "w", encoding="utf-8") as f:
    json.dump(final_chapters, f, ensure_ascii=False, indent=2)

print(" Saved: checkpoints/5_final_chapters.json")

### Whole book level editing

In [ ]:
# Load data

import json

with open("checkpoints/5_final_chapters.json", "r", encoding="utf-8") as f:
    final_chapters = json.load(f)

print(f"Loaded chapters: {len(final_chapters)}")

In [ ]:
#Send each chapter preview of start, middle, end to save tokens/ context space

def smart_preview(text, size=500):
    text_length = len(text)

    # Handle small text safely
    if text_length <= size * 3:
        return text

    start = text[:size]
    middle_start = text_length // 2
    middle = text[middle_start : middle_start + size]
    end = text[-size:]

    return f"{start}\n...\n{middle}\n...\n{end}"

In [ ]:
#Prepare book summary input
#We don’t send full text (too large)
#We send compressed chapter summaries

def build_book_summary(chapters):
    summary = []

    for ch in chapters:
        preview = smart_preview(ch["chapter_text"], size=500)

        summary.append(f"""
Title: {ch['title']}
Theme: {ch['theme']}

Preview:
{preview}
""")

    return "\n\n".join(summary)

In [ ]:
book_summary = build_book_summary(final_chapters)

print(book_summary[:1500])  # preview 

In [ ]:
#Build prompt

def build_editor_prompt(book_summary):
    return f"""
You are a professional book editor.

Analyze this book and provide editorial feedback.

Book Content:
{book_summary}

Tasks:
1. Suggest best reading order of chapters
2. Identify repeated ideas across chapters
3. Highlight tone inconsistencies
4. Suggest chapters that should be merged
5. Provide overall editorial advice

IMPORTANT:
- Be specific and reference chapter titles
- Explain WHY suggestions improve the book

Return ONLY valid JSON:

{{
  "recommended_order": ["..."],
  "repetition_issues": ["..."],
  "tone_issues": ["..."],
  "merge_suggestions": ["..."],
  "flow_improvements": ["..."],
  "editor_notes": "..."
}}
"""

In [ ]:
#Gemini call

prompt = build_editor_prompt(book_summary)

response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
        max_output_tokens=4096
    )
)

raw_output = response.text
print(raw_output[:1000])

In [ ]:
editor_data = extract_json(raw_output)

In [ ]:
#Save checkpoint

with open("checkpoints/6_editor_notes.json", "w", encoding="utf-8") as f:
    json.dump(editor_data, f, ensure_ascii=False, indent=2)

print("Saved: checkpoints/6_editor_notes.json")

### Add front matter and back matter of a book

In [ ]:
# Load data

import json

with open("checkpoints/5_final_chapters.json", "r", encoding="utf-8") as f:
    final_chapters = json.load(f)

with open("checkpoints/6_editor_notes.json", "r", encoding="utf-8") as f:
    editor_notes = json.load(f)

In [ ]:
#Prepare summary input

def build_book_context(chapters):
    context = []

    for ch in chapters:
        preview = smart_preview(ch["chapter_text"], size=500)

        context.append(f"""
            Title: {ch['title']}
            Theme: {ch['theme']}
            Preview:{preview}
            """)

    return "\n\n".join(context)

book_context = build_book_context(final_chapters)

In [ ]:
#Build prompt

def build_front_back_prompt(context, editor_notes):
    return f"""
You are a professional book writer.

Create complete front matter and back matter for a non-fiction book.

Book Structure:
{context}

Editor Insights:
{editor_notes}

Tasks:
1. Write a compelling foreword
2. Write an emotionally engaging and reader-focused introduction  
3. Create a clean table of contents
4. Write a strong conclusion
5. Generate 3-4 key takeaways per chapter
6. Create a glossary of important terms, Use only terms actually appearing in the book
7. Write an author bio (generic if unknown)
8. Compile all fact-check flags into a list

Return ONLY valid JSON:

{{
  "foreword": "...",
  "introduction": "...",
  "table_of_contents": "...",
  "conclusion": "...",
  "key_takeaways": {{
    "Chapter Title": ["...", "..."]
  }},
  "glossary": {{
    "term": "definition"
  }},
  "author_bio": "...",
  "fact_check_list": ["...", "..."]
}}
"""

In [ ]:
#Gemini call

prompt = build_front_back_prompt(book_context, editor_notes)

response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
        max_output_tokens=4096
    )
)

raw_output = response.text
print(raw_output[:1000])

In [ ]:
#Parse json output

front_back_data = extract_json(raw_output)

In [ ]:
#Quick validation

print(front_back_data["table_of_contents"])
print(front_back_data["introduction"])
print(front_back_data["fact_check_list"])

In [ ]:
#Save checkpoint

with open("checkpoints/7_front_back_matter.json", "w", encoding="utf-8") as f:
    json.dump(front_back_data, f, ensure_ascii=False, indent=2)

print("Saved: checkpoints/7_front_back_matter.json")

### Final Book Assembly

In [ ]:
# Load all data

import json

with open("checkpoints/5_final_chapters.json", "r", encoding="utf-8") as f:
    chapters = json.load(f)

with open("checkpoints/6_editor_notes.json", "r", encoding="utf-8") as f:
    editor_notes = json.load(f)

with open("checkpoints/7_front_back_matter.json", "r", encoding="utf-8") as f:
    fb = json.load(f)

In [ ]:
#Apply Chapter Order

def reorder_chapters(chapters, editor_notes):
    if "recommended_order" not in editor_notes:
        return chapters  # fallback

    order = editor_notes["recommended_order"]

    title_map = {ch["title"]: ch for ch in chapters}

    reordered = []
    for title in order:
        if title in title_map:
            reordered.append(title_map[title])

    # add any missing chapters (safety)
    remaining = [ch for ch in chapters if ch["title"] not in order]
    return reordered + remaining


final_chapters = reorder_chapters(chapters, editor_notes)

In [ ]:
#Generate book title and sub-title

def generate_book_title(chapters):
    context = "\n".join([f"{ch['title']}: {ch['theme']}" for ch in chapters])

    prompt = f"""
You are a professional non-fiction book title strategist.

Based on the book structure below, generate:

1. A compelling book title
2. A clear, engaging subtitle

Book Content:
{context}

Guidelines:
- Title should be memorable and concise
- Subtitle should explain value clearly
- Avoid generic phrasing
- Match tone of modern non-fiction bestsellers

Return JSON:
{{
  "title": "...",
  "subtitle": "..."
}}
"""

    response = gemini_client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=100,
            temperature=0.7
        )
    )

    return extract_json(response.text)

In [ ]:
#Apply book title and sub-title

title_data = generate_book_title(final_chapters)

fb["book_title"] = title_data["title"]
fb["book_subtitle"] = title_data["subtitle"]


In [ ]:
print(fb["book_title"])
print(fb["book_subtitle"])

In [ ]:
#Build clickable Table of Contents (TOC)

def build_clickable_toc(chapters):
    toc = "# Table of Contents\n\n"

    for i, ch in enumerate(chapters, 1):
        toc += f'<link href="#chapter_{i}">Chapter {i}: {ch["title"]}</link>\n\n'

    return toc

In [ ]:
fb["table_of_contents"] = build_clickable_toc(final_chapters)

In [ ]:
#Update and aggregate front matter

def build_front_matter(fb):
    return f"""
# {fb.get("book_title", "Untitled Book")}

### {fb.get("book_subtitle", "")}

## Foreword
{fb["foreword"]}

## Introduction
{fb["introduction"]}

## Table of Contents
{fb["table_of_contents"]}
"""

In [ ]:
#Chapters transition generator

def generate_transition(prev_ch, next_ch):
    prompt = f"""
You are a professional book editor.

Write a short transition (2-3 sentences) that smoothly connects two chapters.

Previous Chapter:
Title: {prev_ch['title']}
Theme: {prev_ch['theme']}

Next Chapter:
Title: {next_ch['title']}
Theme: {next_ch['theme']}

Guidelines:
- Ensure logical continuity
- Do NOT repeat content
- Make it feel natural and engaging
- Keep it concise

Output ONLY the transition text.
"""

    response = gemini_client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=120,
        )
    )

    return response.text.strip()

In [ ]:
#Attach transitions in chapter content dict

def add_transitions(chapters):
    for i in range(len(chapters) - 1):
        transition = generate_transition(chapters[i], chapters[i + 1])
        chapters[i]["transition"] = transition

    return chapters

final_chapters = add_transitions(final_chapters)

In [ ]:
#Update chapters with transitions

def build_chapters_with_anchors(chapters):
    text = ""

    for i, ch in enumerate(chapters, 1):
        anchor = f"chapter_{i}"

        text += f"""
<a name="{anchor}"/>

# Chapter {i}: {ch['title']}

{ch['chapter_text']}
"""

        # add transition if exists
        if "transition" in ch:
            text += f"\n\n> *{ch['transition']}*\n"

    return text

In [ ]:
#Aggregate back chapters

def build_back_matter(fb):
    # Key takeaways
    kt_text = ""
    for ch, points in fb["key_takeaways"].items():
        kt_text += f"\n### {ch}\n"
        for p in points:
            kt_text += f"- {p}\n"

    # Glossary
    glossary_text = ""
    for term, definition in fb["glossary"].items():
        glossary_text += f"- **{term}**: {definition}\n"

    # Fact checks
    fact_text = "\n".join([f"- {f}" for f in fb["fact_check_list"]])

    return f"""
---

# Conclusion
{fb["conclusion"]}

---

# Key Takeaways
{kt_text}

---

# Glossary
{glossary_text}

---

# Author Bio
{fb["author_bio"]}

---

# Fact-Check List
{fact_text}
"""

In [ ]:
#Combine front matter, chapters and back matter

book_md = (
    build_front_matter(fb)
    + build_chapters_with_anchors(final_chapters)
    + build_back_matter(fb)
)

In [ ]:
#Save final book checkpoint

import os

os.makedirs("output", exist_ok=True)

with open("output/book.md", "w", encoding="utf-8") as f:
    f.write(book_md)

print("Book saved: output/book.md")

In [ ]:
#Generate book related stats

def generate_stats(chapters, fb):
    total_words = sum(len(ch["chapter_text"].split()) for ch in chapters)
    total_chapters = len(chapters)
    fact_flags = len(fb["fact_check_list"])

    return {
        "total_words": total_words,
        "total_chapters": total_chapters,
        "fact_flags": fact_flags
    }

stats = generate_stats(final_chapters, fb)

with open("output/book_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("Stats saved: output/book_stats.json")
print(stats)

### Generate PDF of Final Book Output

In [ ]:
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak, Image
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER
from reportlab.pdfgen import canvas
import os

# ---------- HEADER / FOOTER ----------
def add_page_numbers(canvas, doc):
    page_num = canvas.getPageNumber()
    text = f"{page_num}"

    canvas.setFont("Helvetica", 10)
    canvas.drawRightString(550, 20, text)

    # Header
    canvas.setFont("Helvetica-Oblique", 9)
    canvas.drawString(40, 820, doc.title if hasattr(doc, "title") else "")


# ---------- MAIN PDF FUNCTION ----------
def generate_pro_pdf(md_text, output_path="output/book.pdf", cover_image=None, cover_back_image=None, title="Book"):
    
    os.makedirs("output", exist_ok=True)

    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        leftMargin=40,
        rightMargin=40,
        topMargin=60,
        bottomMargin=40
    )

    doc.title = title

    styles = getSampleStyleSheet()

    # Custom styles
    title_style = ParagraphStyle(
        name="TitleCustom",
        parent=styles["Title"],
        alignment=TA_CENTER,
        fontSize=24,
        spaceAfter=20
    )

    chapter_style = ParagraphStyle(
        name="Chapter",
        parent=styles["Heading1"],
        spaceBefore=20,
        spaceAfter=12
    )

    normal_style = ParagraphStyle(
        name="NormalCustom",
        parent=styles["Normal"],
        fontName="Helvetica",
        fontSize=11,
        leading=16
    )

    italic_style = ParagraphStyle(
        name="ItalicCustom",
        parent=styles["Italic"],
        fontSize=10,
        leading=14
    )

    elements = []

    # ---------- COVER PAGE ----------
    elements.append(Spacer(1, 5))

    elements.append(Paragraph(title, title_style))
    elements.append(Spacer(1, 5))

    if cover_image and os.path.exists(cover_image):
        elements.append(Image(cover_image, width=6*inch, height=8*inch))
        elements.append(Spacer(1, 20))

    elements.append(Paragraph("Generated by AI Book Pipeline", styles["Normal"]))
    elements.append(PageBreak())

    # ---------- CONTENT ----------
    lines = md_text.split("\n")

    for line in lines:
        line = line.strip()

        if not line:
            elements.append(Spacer(1, 8))
            continue

        # Chapter → page break
        if line.startswith("# Chapter"):
            elements.append(PageBreak())
            elements.append(Paragraph(line.replace("# ", ""), chapter_style))

        # Title
        elif line.startswith("# Conclusion"):
            elements.append(PageBreak())
            elements.append(Paragraph(line[2:], title_style))

        elif line.startswith("# Key Takeaways"):
            elements.append(PageBreak())
            elements.append(Paragraph(line[2:], title_style))

        elif line.startswith("# Glossary"):
            elements.append(PageBreak())
            elements.append(Paragraph(line[2:], title_style))
        
        elif line.startswith("# Author Bio"):
            elements.append(PageBreak())
            elements.append(Paragraph(line[2:], title_style))    

        elif line.startswith("# "):
            elements.append(Paragraph(line[2:], title_style))

        # Subheading
        elif line.startswith("## Table of Contents"):
            elements.append(PageBreak())
            elements.append(Paragraph(line[3:], styles["Heading2"]))

        elif line.startswith("## "):
            elements.append(Paragraph(line[3:], styles["Heading2"]))
        
        elif line.startswith("### "):
            elements.append(Paragraph(line[4:], styles["Heading3"]))

        # Bullet points
        elif line.startswith("- "):
            elements.append(Paragraph(f"• {line[2:]}", normal_style))

        # Blockquote / transition
        elif line.startswith(">"):
            elements.append(Paragraph(line[1:], italic_style))

        # Normal text
        else:
            elements.append(Paragraph(line, normal_style))

    # ---------- END COVER PAGE ----------
    elements.append(Spacer(1, 50))

    if cover_back_image and os.path.exists(cover_back_image):
        elements.append(Image(cover_back_image, width=6*inch, height=8*inch))
        elements.append(Spacer(1, 20))


    # ---------- BUILD PDF ----------
    doc.build(elements, onFirstPage=add_page_numbers, onLaterPages=add_page_numbers)

    print(f" PRO PDF saved: {output_path}")
    

In [ ]:
#Generate sample book cover image

from PIL import Image, ImageDraw, ImageFont

def create_simple_cover(title, output="cover.jpg"):
    img = Image.new("RGB", (800, 1200), color=(30, 30, 60))
    draw = ImageDraw.Draw(img)

    draw.text((100, 500), title, fill=(255, 255, 255))

    img.save(output)
    return output

cover = create_simple_cover("The Intelligence Engine")

In [ ]:
#Call above functions and image

generate_pro_pdf(
    book_md,
    output_path="output/book.pdf",
    cover_image="cover2_online.png",
    cover_back_image="cover2_back_online.png",
    title=fb.get("book_title", "Untitled Book")
)